# 🧪 Multiple Model Comparison

## 🎯 Objective
In this notebook, we evaluate and compare the performance of three different regression algorithms: **Linear Regression**, **XGBoost**, and **Random Forest**. 

We will evaluate each model using the following metrics:
- **MAE** (Mean Absolute Error)
- **RMSE** (Root Mean Squared Error)
- **R² Score** (Coefficient of Determination)
- **MAPE** (Mean Absolute Percentage Error)

### 📥 Data Loading and Preprocessing

In [2]:
import pandas as pd
import numpy as np
import os
import pickle
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression

# Custom function for MAPE calculation
def mean_absolute_percentage_error(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

# Load dataset
df = pd.read_csv('../Synthetic4_local_housePrice.csv')
df['Status'] = df['Status'].str.strip().str.capitalize()
df['Type'] = df['Type'].str.strip().str.capitalize()
df['Price_per_sqft'] = df['Price'] / df['Area']

X = df[['BHK', 'Type', 'Area', 'Status']]
y = df['Price_per_sqft']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define Ordinal Encoder logic
type_order = ['Apartment', 'Row bunglow', 'Independent house']
status_order = ['Under construction', 'Ready to move']

preprocessor = ColumnTransformer(
    transformers=[
        ('ord', OrdinalEncoder(categories=[type_order, status_order]), ['Type', 'Status'])
    ],
    remainder='passthrough'
)

# Ensure artifacts directory exists
if not os.path.exists('../artifacts'):
    os.makedirs('../artifacts')


## 1️⃣ Linear Regression

In [3]:
lr_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

lr_pipeline.fit(X_train, y_train)
lr_preds = lr_pipeline.predict(X_test)


# Save the model
with open('../artifacts/linear_regression_model.pkl', 'wb') as f:
    pickle.dump(lr_pipeline, f)
print("--- Linear Regression Performance ---")
print(f"MAE:      {mean_absolute_error(y_test, lr_preds):.4f}")
print(f"RMSE:     {np.sqrt(mean_squared_error(y_test, lr_preds)):.4f}")
print(f"R² Score: {r2_score(y_test, lr_preds):.4f}")
print(f"MAPE:     {mean_absolute_percentage_error(y_test, lr_preds):.4f}%")

--- Linear Regression Performance ---
MAE:      178.0863
RMSE:     220.7881
R² Score: 0.8769
MAPE:     3.6628%


In [6]:
# Example Prediction: Linear Regression
import pandas as pd
sample_input = pd.DataFrame([[2, 'Apartment', 1000, 'Ready to move']], 
                           columns=['BHK', 'Type', 'Area', 'Status'])
prediction = lr_pipeline.predict(sample_input)[0]
print(f"Predicted Price per sqft: {prediction:.2f}")
print(f"Estimated Total Price: {prediction * 1000:.2f}")

Predicted Price per sqft: 4442.38
Estimated Total Price: 4442377.50


## 2️⃣ XGBoost Regressor

In [4]:
xgb_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', xgb.XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42))
])

xgb_pipeline.fit(X_train, y_train)
xgb_preds = xgb_pipeline.predict(X_test)


# Save the model
with open('../artifacts/xgboost_model.pkl', 'wb') as f:
    pickle.dump(xgb_pipeline, f)
print("--- XGBoost Performance ---")
print(f"MAE:      {mean_absolute_error(y_test, xgb_preds):.4f}")
print(f"RMSE:     {np.sqrt(mean_squared_error(y_test, xgb_preds)):.4f}")
print(f"R² Score: {r2_score(y_test, xgb_preds):.4f}")
print(f"MAPE:     {mean_absolute_percentage_error(y_test, xgb_preds):.4f}%")

--- XGBoost Performance ---
MAE:      180.4465
RMSE:     224.0447
R² Score: 0.8732
MAPE:     3.7134%


In [7]:
# Example Prediction: XGBoost
import pandas as pd
sample_input = pd.DataFrame([[2, 'Apartment', 1000, 'Ready to move']], 
                           columns=['BHK', 'Type', 'Area', 'Status'])
prediction = xgb_pipeline.predict(sample_input)[0]
print(f"Predicted Price per sqft: {prediction:.2f}")
print(f"Estimated Total Price: {prediction * 1000:.2f}")

Predicted Price per sqft: 4407.32
Estimated Total Price: 4407321.00


## 3️⃣ Random Forest Regressor

In [5]:
rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
])

rf_pipeline.fit(X_train, y_train)
rf_preds = rf_pipeline.predict(X_test)


# Save the model
with open('../artifacts/random_forest_model.pkl', 'wb') as f:
    pickle.dump(rf_pipeline, f)
print("--- Random Forest Performance ---")
print(f"MAE:      {mean_absolute_error(y_test, rf_preds):.4f}")
print(f"RMSE:     {np.sqrt(mean_squared_error(y_test, rf_preds)):.4f}")
print(f"R² Score: {r2_score(y_test, rf_preds):.4f}")
print(f"MAPE:     {mean_absolute_percentage_error(y_test, rf_preds):.4f}%")

--- Random Forest Performance ---
MAE:      209.6270
RMSE:     261.8168
R² Score: 0.8269
MAPE:     4.3106%


In [8]:
# Example Prediction: Random Forest
import pandas as pd
sample_input = pd.DataFrame([[2, 'Apartment', 1000, 'Ready to move']], 
                           columns=['BHK', 'Type', 'Area', 'Status'])
prediction = rf_pipeline.predict(sample_input)[0]
print(f"Predicted Price per sqft: {prediction:.2f}")
print(f"Estimated Total Price: {prediction * 1000:.2f}")

Predicted Price per sqft: 4454.42
Estimated Total Price: 4454424.78
